# 04 — Cross-File Generalization

Two experiments testing how models generalize to unseen attack types.

**Experiment A** — Train: Web + DDoS  →  Test: PortScan

**Experiment B** — Train: All except Infiltration  →  Test: Infiltration

In [1]:
%run 00_data_loader.ipynb

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

Config loaded
preprocess_binary() defined
load_all_datasets() defined


## 1. Load All Datasets

In [2]:
datasets = load_all_datasets()
print(f"Loaded: {list(datasets.keys())}")

Loading 8 dataset(s): ['monday', 'bruteforce', 'dos', 'web_attacks', 'infiltration', 'botnet', 'portscan', 'ddos']

  [monday]  Monday-WorkingHours.pcap_ISCX.csv
           raw (529918, 79)  ->  clean (502650, 81)  (attack rate 0.0%)
  [bruteforce]  Tuesday-WorkingHours.pcap_ISCX.csv
           raw (445909, 79)  ->  clean (421626, 81)  (attack rate 2.2%)
  [dos]  Wednesday-workingHours.pcap_ISCX.csv
           raw (692703, 79)  ->  clean (610492, 81)  (attack rate 31.7%)
  [web_attacks]  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
           raw (170366, 79)  ->  clean (164179, 81)  (attack rate 1.3%)
  [infiltration]  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
           raw (288602, 79)  ->  clean (252790, 81)  (attack rate 0.0%)
  [botnet]  Friday-WorkingHours-Morning.pcap_ISCX.csv
           raw (191033, 79)  ->  clean (184044, 81)  (attack rate 1.1%)
  [portscan]  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
           raw (286467, 79)  ->  clea

## 2. Helper

In [3]:
def run_experiment(train_df, test_df, experiment_name):
    print(f"\n{'='*60}")
    print(f"  {experiment_name}")
    print(f"  Train: {len(train_df):,}   Test: {len(test_df):,}")
    print(f"  Test attack rate: {test_df['Label_Binary'].mean()*100:.1f}%")
    print(f"{'='*60}")

    drop_cols = ["Label", "Label_Binary", "Source_File"]
    X_train = train_df.drop(columns=drop_cols).replace([np.inf, -np.inf], np.nan)
    X_test  = test_df.drop(columns=drop_cols).replace([np.inf, -np.inf], np.nan)
    y_train = train_df["Label_Binary"]
    y_test  = test_df["Label_Binary"]

    medians = X_train.median(numeric_only=True)
    X_train = X_train.fillna(medians)
    X_test  = X_test.fillna(medians)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    results = []
    for name, model, Xtr, Xte in [
        ("Logistic Regression", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42), X_train_sc, X_test_sc),
        ("Random Forest",       RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42), X_train, X_test),
    ]:
        model.fit(Xtr, y_train)
        y_pred = model.predict(Xte)
        y_prob = model.predict_proba(Xte)[:, 1]
        print(f"\n{name} — Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
        results.append({
            "Model":     name,
            "Accuracy":  accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall":    recall_score(y_test, y_pred, zero_division=0),
            "F1":        f1_score(y_test, y_pred, zero_division=0),
            "ROC_AUC":   roc_auc_score(y_test, y_prob),
        })

    return pd.DataFrame(results)

## 3. Experiment A — Web + DDoS  →  PortScan

In [4]:
train_A = pd.concat([datasets["web_attacks"], datasets["ddos"]], ignore_index=True)
results_A = run_experiment(train_A, datasets["portscan"], "Exp A: Web+DDoS → PortScan")
results_A.round(4)


  Exp A: Web+DDoS → PortScan
  Train: 387,261   Test: 213,777
  Test attack rate: 42.4%

Logistic Regression — Confusion Matrix:
 [[119656   3427]
 [ 90553    141]]

Random Forest — Confusion Matrix:
 [[123074      9]
 [ 90626     68]]


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.5604,0.0395,0.0016,0.0030,0.7211
1,Random Forest,0.5760,0.8831,0.0007,0.0015,0.6707


## 4. Experiment B — All Except Infiltration  →  Infiltration

In [5]:
train_B = pd.concat([v for k, v in datasets.items() if k != "infiltration"], ignore_index=True)
results_B = run_experiment(train_B, datasets["infiltration"], "Exp B: All → Infiltration")
results_B.round(4)


  Exp B: All → Infiltration
  Train: 2,319,850   Test: 252,790
  Test attack rate: 0.0%

Logistic Regression — Confusion Matrix:
 [[202503  50251]
 [    26     10]]

Random Forest — Confusion Matrix:
 [[251666   1088]
 [    36      0]]


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.8011,0.0002,0.2778,0.0004,0.3344
1,Random Forest,0.9956,0.0000,0.0000,0.0000,0.6396


## 5. Side-by-Side Comparison

In [6]:
results_A["Experiment"] = "A: Web+DDoS → PortScan"
results_B["Experiment"] = "B: All → Infiltration"
pd.concat([results_A, results_B])[["Experiment","Model","Accuracy","Precision","Recall","F1","ROC_AUC"]].round(4)

,Experiment,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,A: Web+DDoS → PortScan,Logistic Regression,0.5604,0.0395,0.0016,0.0030,0.7211
1,A: Web+DDoS → PortScan,Random Forest,0.5760,0.8831,0.0007,0.0015,0.6707
0,B: All → Infiltration,Logistic Regression,0.8011,0.0002,0.2778,0.0004,0.3344
1,B: All → Infiltration,Random Forest,0.9956,0.0000,0.0000,0.0000,0.6396


## Cross-Dataset Generalization: CICIDS2017 vs CSE-CIC-IDS2018

The earlier experiments tested generalization across different CICIDS2017 attack files. After adding CSE-CIC-IDS2018, I also tested whether models trained on one dataset year could generalize to another dataset year. This is a harder and more realistic test because real intrusion detection systems may face newer traffic patterns that are different from the training data.